# Moving Average Crossover Strategy Backtest

This notebook demonstrates how to backtest the Moving Average Crossover strategy using Nautilus Trader.

In [1]:
import os
import sys
from datetime import datetime, timedelta
from decimal import Decimal

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.backtest.engine import BacktestEngineConfig
from nautilus_trader.backtest.data import BacktestDataConfig
from nautilus_trader.backtest.modules import FXRolloverInterestModule
from nautilus_trader.config import LoggingConfig
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.enums import AccountType
from nautilus_trader.model.enums import BarAggregation
from nautilus_trader.model.enums import PriceType
from nautilus_trader.model.identifiers import InstrumentId
from nautilus_trader.model.identifiers import Symbol
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.model.objects import Money
from nautilus_trader.model.data import BarType
from nautilus_trader.test.test_kit import UNIX_EPOCH

# Add the src directory to the path so we can import our strategy
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from src.strategies.moving_average_crossover import MovingAverageCrossover, MovingAverageCrossoverConfig

ModuleNotFoundError: No module named 'nautilus_trader.backtest.data'

## Load Data

First, we'll load the data for our backtest. We'll use the Binance data downloaded with the `scripts/download_data.py` script.

In [ ]:
# Define the symbol and timeframe
symbol = "BTCUSDT"
timeframe = "1h"

# Path to the data file
data_path = f"../../data/catalog/binance/{symbol}/{symbol}_{timeframe}.csv"

# Check if the data file exists
if not os.path.exists(data_path):
    print(f"Data file not found: {data_path}")
    print("Please run the scripts/download_data.py script to download the data.")
    # You can run the script directly from the notebook
    !python ../../scripts/download_data.py --symbols BTCUSDT --timeframes 1h

# Load the data
df = pd.read_csv(data_path, index_col=0, parse_dates=True)

# Display the first few rows
df.head()

## Prepare Data for Backtest

Now we'll prepare the data for the backtest by converting it to the format expected by Nautilus Trader.

In [ ]:
# Convert the data to the format expected by Nautilus Trader
df_bars = pd.DataFrame()
df_bars["open"] = df["open"]
df_bars["high"] = df["high"]
df_bars["low"] = df["low"]
df_bars["close"] = df["close"]
df_bars["volume"] = df["volume"]

# Define the instrument
instrument_id = InstrumentId(Symbol(symbol), Venue("BINANCE"))

# Define the bar type
if timeframe == "1m":
    bar_aggregation = BarAggregation.MINUTE
    bar_spec = 1
elif timeframe == "5m":
    bar_aggregation = BarAggregation.MINUTE
    bar_spec = 5
elif timeframe == "15m":
    bar_aggregation = BarAggregation.MINUTE
    bar_spec = 15
elif timeframe == "1h":
    bar_aggregation = BarAggregation.HOUR
    bar_spec = 1
elif timeframe == "4h":
    bar_aggregation = BarAggregation.HOUR
    bar_spec = 4
elif timeframe == "1d":
    bar_aggregation = BarAggregation.DAY
    bar_spec = 1
else:
    raise ValueError(f"Unsupported timeframe: {timeframe}")

bar_type = BarType(
    instrument_id=instrument_id,
    bar_spec=bar_spec,
    aggregation=bar_aggregation,
    price_type=PriceType.LAST,
)

## Configure and Run Backtest

Now we'll configure and run the backtest using the Moving Average Crossover strategy.

In [ ]:
# Define the backtest configuration
config = BacktestEngineConfig(
    logging=LoggingConfig(log_level="INFO"),
    run_analysis=True,
)

# Create the backtest engine
engine = BacktestEngine(config=config)

# Add the instrument
engine.add_instrument(
    instrument_id=instrument_id,
    price_precision=2,
    size_precision=6,
    price_increment=Decimal("0.01"),
    size_increment=Decimal("0.000001"),
    multiplier=Decimal("1"),
    lot_size=Decimal("1"),
    max_quantity=Decimal("1000"),
    min_quantity=Decimal("0.000001"),
    margin_init=Decimal("0"),
    margin_maint=Decimal("0"),
    base_currency=USD,
    quote_currency=USD,
    is_inverse=False,
    maker_fee=Decimal("0.001"),
    taker_fee=Decimal("0.001"),
)

# Add the data
engine.add_data(
    bars=df_bars,
    bar_type=bar_type,
    start_time=df.index[0],
    end_time=df.index[-1],
)

# Add a trading venue
engine.add_venue(
    venue=Venue("BINANCE"),
    oms_type="NETTING",
    account_type=AccountType.MARGIN,
    base_currency=USD,
    starting_balances=[Money(1_000_000, USD)],
)

# Create the strategy configuration
strategy_config = MovingAverageCrossoverConfig(
    instrument_id=instrument_id,
    bar_type=bar_type,
    fast_ema_period=10,
    slow_ema_period=20,
    trade_size=Decimal("0.1"),
)

# Create and add the strategy
strategy = MovingAverageCrossover(config=strategy_config)
engine.add_strategy(strategy=strategy)

# Run the backtest
engine.run()

## Analyze Results

Now we'll analyze the results of the backtest.

In [ ]:
# Get the performance statistics
performance = engine.trader.generate_account_report(Venue("BINANCE"))
print(performance)

In [ ]:
# Plot the equity curve
equity_curve = engine.trader.generate_equity_curve(Venue("BINANCE"))
plt.figure(figsize=(12, 6))
plt.plot(equity_curve.index, equity_curve["equity"])
plt.title("Equity Curve")
plt.xlabel("Time")
plt.ylabel("Equity (USD)")
plt.grid(True)
plt.show()

In [ ]:
# Get the order fills
order_fills = engine.trader.generate_order_fills_report()
order_fills.head()

## Optimize Strategy Parameters

Now we'll optimize the strategy parameters to find the best combination of fast and slow EMA periods.

In [ ]:
# Define the parameter ranges
fast_periods = range(5, 21, 5)  # 5, 10, 15, 20
slow_periods = range(20, 51, 10)  # 20, 30, 40, 50

# Initialize results storage
results = []

# Run backtests for each parameter combination
for fast_period in fast_periods:
    for slow_period in slow_periods:
        if fast_period >= slow_period:
            continue  # Skip invalid combinations
        
        # Create a new engine for each backtest
        engine = BacktestEngine(config=config)
        
        # Add the instrument
        engine.add_instrument(
            instrument_id=instrument_id,
            price_precision=2,
            size_precision=6,
            price_increment=Decimal("0.01"),
            size_increment=Decimal("0.000001"),
            multiplier=Decimal("1"),
            lot_size=Decimal("1"),
            max_quantity=Decimal("1000"),
            min_quantity=Decimal("0.000001"),
            margin_init=Decimal("0"),
            margin_maint=Decimal("0"),
            base_currency=USD,
            quote_currency=USD,
            is_inverse=False,
            maker_fee=Decimal("0.001"),
            taker_fee=Decimal("0.001"),
        )
        
        # Add the data
        engine.add_data(
            bars=df_bars,
            bar_type=bar_type,
            start_time=df.index[0],
            end_time=df.index[-1],
        )
        
        # Add a trading venue
        engine.add_venue(
            venue=Venue("BINANCE"),
            oms_type="NETTING",
            account_type=AccountType.MARGIN,
            base_currency=USD,
            starting_balances=[Money(1_000_000, USD)],
        )
        
        # Create the strategy with the current parameters
        strategy_config = MovingAverageCrossoverConfig(
            instrument_id=instrument_id,
            bar_type=bar_type,
            fast_ema_period=fast_period,
            slow_ema_period=slow_period,
            trade_size=Decimal("0.1"),
        )
        
        strategy = MovingAverageCrossover(config=strategy_config)
        engine.add_strategy(strategy=strategy)
        
        # Run the backtest
        engine.run()
        
        # Get the performance statistics
        performance = engine.trader.generate_account_report(Venue("BINANCE"))
        
        # Store the results
        results.append({
            "fast_period": fast_period,
            "slow_period": slow_period,
            "return": performance.return_pct,
            "sharpe": performance.sharpe_ratio,
            "max_drawdown": performance.max_drawdown_pct,
            "win_rate": performance.win_rate,
        })
        
        print(f"Completed backtest: Fast EMA = {fast_period}, Slow EMA = {slow_period}, Return = {performance.return_pct:.2f}%")

# Convert results to DataFrame
results_df = pd.DataFrame(results)
results_df.sort_values(by="return", ascending=False, inplace=True)
results_df.head(10)

In [ ]:
# Plot the results
plt.figure(figsize=(12, 8))

# Create a pivot table for the heatmap
heatmap_data = results_df.pivot(index="fast_period", columns="slow_period", values="return")

# Plot the heatmap
plt.imshow(heatmap_data, cmap="viridis", aspect="auto")
plt.colorbar(label="Return (%)")

# Set the tick labels
plt.xticks(range(len(heatmap_data.columns)), heatmap_data.columns)
plt.yticks(range(len(heatmap_data.index)), heatmap_data.index)

# Add labels and title
plt.xlabel("Slow EMA Period")
plt.ylabel("Fast EMA Period")
plt.title("Moving Average Crossover Strategy Returns")

# Add the values to the heatmap
for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        value = heatmap_data.iloc[i, j]
        if not np.isnan(value):
            plt.text(j, i, f"{value:.1f}%", ha="center", va="center", color="white")

plt.tight_layout()
plt.show()

## Conclusion

In this notebook, we've demonstrated how to backtest and optimize a Moving Average Crossover strategy using Nautilus Trader. We've seen how to:

1. Load and prepare data for backtesting
2. Configure and run a backtest
3. Analyze the results
4. Optimize strategy parameters

The best parameter combination based on our optimization is shown in the sorted results table above.